In [36]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import time

In [37]:
df = pd.read_csv('../data/668f9d46-ba94-4582-8496-d5ac9a7d2ce2.csv')

# check if ticker_code is a 4-digit number
def is_four_digit_number(x):
    if isinstance(x, str):
        return x.isdigit() and len(x) == 4
    return False

# filter data
df_filtered = df[df['TICKER_CODE'].apply(is_four_digit_number)]    

df_filtered

,DATE,COUNT,TOTAL_SALES,TICKER_CODE
0,2013-01-01,20,73761,2138
1,2013-01-01,12,85905,2157
2,2013-01-01,2,4070,2193
3,2013-01-01,35,120393,2211
4,2013-01-01,2,6000,2267
...,...,...,...,...
1107795,2026-03-31,3364,11895353,4478
1107796,2026-04-30,1724,5072281,3994
1107797,2026-04-30,1695,5980035,4478
1107798,2026-05-31,39,149150,3994


In [38]:
# convert DATE to datetime
df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'])

# filter data from April 2021
end_date = pd.to_datetime('2025-07-01')
df_filtered = df_filtered[df_filtered['DATE'] <= end_date]

# add 'year-month' column for monthly aggregation
df_filtered['year-month'] = df_filtered['DATE'].dt.to_period('M')

# sum by TICKER_CODE and year-month
monthly_total = df_filtered.groupby(['year-month', 'TICKER_CODE'])['TOTAL_SALES'].sum().reset_index()
monthly_total.rename(columns={'year-month': 'DATE'}, inplace=True)

# calculate growth rate
monthly_total['prev_month_sales'] = monthly_total.groupby('TICKER_CODE')['TOTAL_SALES'].shift(1)
monthly_total['growth_rate'] = (monthly_total['TOTAL_SALES'] - monthly_total['prev_month_sales']) / monthly_total['prev_month_sales']
monthly_total = monthly_total.dropna(subset=['growth_rate'])

# growth rateのYoY
monthly_total['prev_year_growth_rate'] = monthly_total.groupby('TICKER_CODE')['growth_rate'].shift(12)
monthly_total['YoY'] = monthly_total['growth_rate'] - monthly_total['prev_year_growth_rate']
monthly_total = monthly_total.dropna(subset=['YoY'])

# exclude the latest month
latest_month = monthly_total['DATE'].max()
monthly_total = monthly_total[monthly_total['DATE'] != latest_month]

# convert DATE to string
monthly_total['DATE'] = monthly_total['DATE'].astype(str)

monthly_total


C:\Users\hh080\AppData\Local\Temp\ipykernel_14104\3385790853.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['DATE'] = pd.to_datetime(df_filtered['DATE'])
C:\Users\hh080\AppData\Local\Temp\ipykernel_14104\3385790853.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['year-month'] = df_filtered['DATE'].dt.to_period('M')


,DATE,TICKER_CODE,TOTAL_SALES,prev_month_sales,growth_rate,prev_year_growth_rate,YoY
2904,2014-02,2138,2553303,3.470893e+06,-0.264367,0.195360,-0.459727
2905,2014-02,2157,1929934,2.673437e+06,-0.278108,-0.129991,-0.148116
2906,2014-02,2193,4320986,4.141117e+06,0.043435,0.034751,0.008683
2907,2014-02,2211,2605782,2.872552e+06,-0.092869,-0.007298,-0.085571
2908,2014-02,2267,3019725,2.401000e+05,11.576947,6.187059,5.389888
...,...,...,...,...,...,...,...
34932,2025-05,9974,327291989,3.090010e+08,0.059194,0.049375,0.009819
34933,2025-05,9983,1720150602,1.441297e+09,0.193474,0.045424,0.148050
34934,2025-05,9984,4053911322,5.968252e+09,-0.320754,-0.022500,-0.298254
34935,2025-05,9989,403450209,3.820801e+08,0.055931,0.118081,-0.062150


In [39]:
# 不要なカラムの削除
monthly_total = monthly_total.drop(columns=['TOTAL_SALES', 'prev_month_sales', 'growth_rate', 'prev_year_growth_rate'])

# TICKER_CODEとDATEで並び替え
monthly_total = monthly_total.sort_values(by=['TICKER_CODE', 'DATE'])

monthly_total

,DATE,TICKER_CODE,YoY
2904,2014-02,2138,-0.459727
3134,2014-03,2138,-0.330166
3363,2014-04,2138,0.525741
3593,2014-05,2138,-0.209207
3822,2014-06,2138,-0.142646
...,...,...,...
33984,2025-01,9997,-0.128289
34222,2025-02,9997,0.129865
34460,2025-03,9997,0.198623
34698,2025-04,9997,-0.287542


In [40]:
quantile_data = []

# YoYを用いたquantileの作成
for date in monthly_total['DATE'].unique():
    date_data = monthly_total[monthly_total['DATE'] == date].copy()  # .copy()を追加

    date_data.loc[:, 'quantile_YoY'] = pd.qcut(date_data['YoY'], 4, labels=range(1, 4 + 1))

    quantile_data.append(date_data)

quantile_df = pd.concat(quantile_data)

quantile_df.dropna(inplace=True)

quantile_df


c:\Users\hh080\repositories\Nowcast\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in multiply
  lerp_interpolation = asanyarray(add(a, diff_b_a * t, out=out))
c:\Users\hh080\repositories\Nowcast\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in multiply
  lerp_interpolation = asanyarray(add(a, diff_b_a * t, out=out))
c:\Users\hh080\repositories\Nowcast\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4669: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\hh080\repositories\Nowcast\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in multiply
  lerp_interpolation = asanyarray(add(a, diff_b_a * t, out=out))


,DATE,TICKER_CODE,YoY,quantile_YoY
2904,2014-02,2138,-0.459727,1
2905,2014-02,2157,-0.148116,1
2906,2014-02,2193,0.008683,3
2907,2014-02,2211,-0.085571,2
2908,2014-02,2267,5.389888,4
...,...,...,...,...
34932,2025-05,9974,0.009819,3
34933,2025-05,9983,0.148050,4
34934,2025-05,9984,-0.298254,1
34935,2025-05,9989,-0.062150,2


In [41]:
portfolio_weights = []
for date in quantile_df['DATE'].unique():
    date_quantiles = quantile_df[quantile_df['DATE'] == date]
    for quantile in range(1, 4 + 1):
        quantile_stocks = date_quantiles[date_quantiles['quantile_YoY'] == quantile]
        if not quantile_stocks.empty:
            weight = 1.0 / len(quantile_stocks)
            for _, row in quantile_stocks.iterrows():
                portfolio_weights.append({'DATE': date, 'TICKER_CODE': row['TICKER_CODE'], f'quantile_{quantile}': weight})
    
    portfolio_weights_df = pd.DataFrame(portfolio_weights)
    portfolio_weights_df = portfolio_weights_df.fillna(0)

portfolio_weights_df

,DATE,TICKER_CODE,quantile_1,quantile_2,quantile_3,quantile_4
0,2014-02,2138,0.018182,0.0,0.0,0.000000
1,2014-02,2157,0.018182,0.0,0.0,0.000000
2,2014-02,2484,0.018182,0.0,0.0,0.000000
3,2014-02,2659,0.018182,0.0,0.0,0.000000
4,2014-02,2695,0.018182,0.0,0.0,0.000000
...,...,...,...,...,...,...
31660,2025-05,9733,0.000000,0.0,0.0,0.016667
31661,2025-05,9766,0.000000,0.0,0.0,0.016667
31662,2025-05,9900,0.000000,0.0,0.0,0.016667
31663,2025-05,9983,0.000000,0.0,0.0,0.016667


In [42]:
# load price data
price_data = pd.read_csv('../data/new_price_data.csv')

In [43]:
price_data = price_data.drop(columns=['dividends'])

price_data = price_data[price_data['DATE'] <= '2025-05-01']
price_data = price_data[price_data['DATE'] >= '2014-04-01']

In [47]:
# DATEの形式を統一
portfolio_weights_df['DATE'] = pd.to_datetime(portfolio_weights_df['DATE']).dt.strftime('%Y-%m')
price_data['DATE'] = pd.to_datetime(price_data['DATE']).dt.strftime('%Y-%m')

# TICKER_CODEの型を統一（文字列に変換）
portfolio_weights_df['TICKER_CODE'] = portfolio_weights_df['TICKER_CODE'].astype(str)
price_data['TICKER_CODE'] = price_data['TICKER_CODE'].astype(str)

# リターンマトリックスの作成
returns_matrix = price_data.pivot(index='DATE', columns='TICKER_CODE', values='monthly_return')

# 各クォンタイルのリターン計算
portfolio_returns = pd.DataFrame(index=returns_matrix.index)

# クォンタイル列の自動検出
quantile_cols = [col for col in portfolio_weights_df.columns if col.startswith('quantile_')]

for quantile_col in quantile_cols:
    # ウェイトの取得
    weights = portfolio_weights_df.pivot(index='DATE', columns='TICKER_CODE', values=quantile_col)

    # インデックスの整合性確認
    common_dates = weights.index.intersection(returns_matrix.index)

    if len(common_dates) == 0:
        print(f"Warning: No common dates found between weights and returns for {quantile_col}")
        continue

    # リターンの計算
    # 1. 共通の日付でデータを抽出
    weights_common = weights.loc[common_dates]
    returns_common = returns_matrix.loc[common_dates]
    
    # 2. 共通のカラム（銘柄）を取得
    common_columns = weights_common.columns.intersection(returns_common.columns)
    
    if len(common_columns) == 0:
        print("Warning: No common columns found between weights and returns")
        continue
        
    weights_common = weights_common[common_columns]
    returns_common = returns_common[common_columns]
    
    # 3. NaNを0に置換
    weights_common = weights_common.fillna(0)
    returns_common = returns_common.fillna(0)
    
    weighted_returns = weights_common * returns_common
    
    # 5. 合計を計算
    returns = weighted_returns.sum(axis=1)

    portfolio_returns[quantile_col] = returns

# NaNを除去
portfolio_returns = portfolio_returns.dropna()

portfolio_returns


,quantile_1,quantile_2,quantile_3,quantile_4
DATE,,,,
2014-05,0.046402,0.006491,0.037768,0.027808
2014-06,0.055422,0.046975,0.040501,0.046533
2014-07,0.015033,0.021686,0.022119,0.042500
2014-08,0.016537,-0.005296,-0.009665,-0.005560
2014-09,0.031709,0.017112,0.013789,0.010520
...,...,...,...,...
2025-01,0.015557,0.015672,0.012031,0.018319
2025-02,-0.040090,-0.006428,-0.023280,-0.010181
2025-03,0.020930,0.016902,0.029529,0.013644


In [48]:

portfolio_returns.to_csv('../data/output/portfolio_returns_1325_sales.csv')

In [45]:
# set date index
portfolio_returns.index = pd.to_datetime(portfolio_returns.index)


In [46]:
def plot_portfolio_returns(portfolio_returns, compare_month, ax):
    for col in portfolio_returns.columns:
        # 累計リターンを計算
        cumulative_returns = (1 + portfolio_returns[col]).cumprod()
        
        # 初月が1になるように調整
        cumulative_returns = cumulative_returns / cumulative_returns.iloc[0]

        ax.plot(portfolio_returns.index, 
        cumulative_returns, 
        label=col)

    ax.set_title(f'{compare_month-1}M')
    ax.set_xlabel('date')
    ax.set_ylabel('cumulative return')
    ax.legend()
    ax.grid(True)
    ax.tick_params(axis='x', rotation=45)

# visualize growth rate
plot_portfolio_returns(portfolio_returns)


TypeError: plot_portfolio_returns() missing 2 required positional arguments: 'compare_month' and 'ax'